# 1. Libraries and paths

In [1]:
import pandas as pd
import re
from Bio import SeqIO

In [2]:
input_path = "../general_input/"
train_path = input_path + "train/"
test_path = input_path + "test/"

train_fasta = train_path + "train_sequences.fasta"
test_fasta = test_path + "testsuperset.fasta"

# 2. Train file with all the important information

In [3]:
# Regex to detect the different fields in the fasta headers
re_db = re.compile(r'^(?P<Database>\w+)\|(?P<ID>[^|]+)\|(?P<Name>[^\s]+)\s*(?P<Full_Name>.*)$')
re_ox = re.compile(r'\bOX=(?P<TaxID>\d+)\b')
re_gn = re.compile(r'\bGN=(?P<Gen>[^\s]+)\b')
re_pe = re.compile(r'\bPE=(?P<PE>\d+)\b')
re_sv = re.compile(r'\bSV=(?P<SV>\d+)\b')

fieldnames = ["Database", "ID", "Name", "Full_Name", "Organism", "TaxID", "Gen", "PE", "SV", "Sequence", "Length"]

rows = []

for record in SeqIO.parse(train_fasta, "fasta"):
    header = record.description

    if " OS=" in header:
        left, meta = header.split(" OS=", 1)   # meta starts with "Homo sapiens ... OX=..."
        organism_part = meta  # has "Homo sapiens ... OX=9606 GN=... PE=... SV=..."
    else:
        left = header
        organism_part = ""  # no metadata

    # extract Database, ID, Name y Full_Name from left part
    m = re_db.match(left)
    if m:
        info = m.groupdict()
        # Remove spaces
        info["Full_Name"] = info["Full_Name"].strip() if info["Full_Name"] else None
    else:
        info = {k: None for k in ["Database", "ID", "Name", "Full_Name"]}

    # extract Organism
    if organism_part:
        if " OX=" in organism_part:
            organism = organism_part.split(" OX=", 1)[0].strip()
        else:
            # si no hay OX=, tomamos hasta el primer ' GN=' o ' PE=' o ' SV=' o el final
            organism = re.split(r'\s+(GN=|PE=|SV=)', organism_part, maxsplit=1)[0].strip()
        organism = organism if organism != "" else None
    else:
        organism = None

    # search OX, GN, PE, SV in 'organism_part'
    ox_m = re_ox.search(organism_part)
    gn_m = re_gn.search(organism_part)
    pe_m = re_pe.search(organism_part)
    sv_m = re_sv.search(organism_part)

    taxid = ox_m.group("TaxID") if ox_m else None
    gen = gn_m.group("Gen") if gn_m else None
    pe = pe_m.group("PE") if pe_m else None
    sv = sv_m.group("SV") if sv_m else None

    seq = str(record.seq)
    length = len(seq)

    row = {
        "Database": info.get("Database"),
        "ID": info.get("ID"),
        "Name": info.get("Name"),
        "Full_Name": info.get("Full_Name"),
        "Organism": organism,
        "TaxID": int(taxid),
        "Gen": gen,
        "PE": int(pe),
        "SV": int(sv),
        "Sequence": seq,
        "Length": length
    }

    rows.append(row)
    
df_complete = pd.DataFrame(rows, columns=fieldnames)

In [4]:
df_complete.head()

,Database,ID,Name,Full_Name,Organism,TaxID,Gen,PE,SV,Sequence,Length
0,sp,A0A0C5B5G6,MOTSC_HUMAN,Mitochondrial-derived peptide MOTS-c,Homo sapiens,9606,MT-RNR1,1,1,MRWQEMGYIFYPRKLR,16
1,sp,A0JNW5,BLT3B_HUMAN,Bridge-like lipid transfer protein family memb...,Homo sapiens,9606,BLTP3B,1,2,MAGIIKKQILKHLSRFTKNLSPDKINLSTLKGEGELKNLELDEEVL...,1464
2,sp,A0JP26,POTB3_HUMAN,POTE ankyrin domain family member B3,Homo sapiens,9606,POTEB3,1,2,MVAEVCSMPAASAVKKPFDLRSKMGKWCHHRFPCCRGSGKSNMGTS...,581
3,sp,A0PK11,CLRN2_HUMAN,Clarin-2,Homo sapiens,9606,CLRN2,1,1,MPGWFKKAWYGLASLLSFSSFILIIVALVVPHWLSGKILCQTGVDL...,232
4,sp,A1A4S6,RHG10_HUMAN,Rho GTPase-activating protein 10,Homo sapiens,9606,ARHGAP10,1,1,MGLQPLEFSDCYLDSPWFRERIRAHEAELERTNKFIKELIKDGKNL...,786


In [5]:
# Change column names
new_columns = ["db", "id", "name", "full_name", "organism", "taxon", "gen", "PE", "SV", "sequence", "length"]
df_complete.columns = new_columns

df_complete.head()

,db,id,name,full_name,organism,taxon,gen,PE,SV,sequence,length
0,sp,A0A0C5B5G6,MOTSC_HUMAN,Mitochondrial-derived peptide MOTS-c,Homo sapiens,9606,MT-RNR1,1,1,MRWQEMGYIFYPRKLR,16
1,sp,A0JNW5,BLT3B_HUMAN,Bridge-like lipid transfer protein family memb...,Homo sapiens,9606,BLTP3B,1,2,MAGIIKKQILKHLSRFTKNLSPDKINLSTLKGEGELKNLELDEEVL...,1464
2,sp,A0JP26,POTB3_HUMAN,POTE ankyrin domain family member B3,Homo sapiens,9606,POTEB3,1,2,MVAEVCSMPAASAVKKPFDLRSKMGKWCHHRFPCCRGSGKSNMGTS...,581
3,sp,A0PK11,CLRN2_HUMAN,Clarin-2,Homo sapiens,9606,CLRN2,1,1,MPGWFKKAWYGLASLLSFSSFILIIVALVVPHWLSGKILCQTGVDL...,232
4,sp,A1A4S6,RHG10_HUMAN,Rho GTPase-activating protein 10,Homo sapiens,9606,ARHGAP10,1,1,MGLQPLEFSDCYLDSPWFRERIRAHEAELERTNKFIKELIKDGKNL...,786


In [6]:
# Read the train_terms file and add the GO terms to the dataframe as a list in a new column
train_terms_file = train_path + "train_terms.tsv"
df_terms = pd.read_csv(train_terms_file, sep="\t", names=["id", "go_terms", "aspect"])
df_terms['go_terms'] = df_terms['go_terms'].apply(lambda x: [x])

In [7]:
df_terms.head()

,id,go_terms,aspect
0,EntryID,[term],aspect
1,Q5W0B1,[GO:0000785],C
2,Q5W0B1,[GO:0004842],F
3,Q5W0B1,[GO:0051865],P
4,Q5W0B1,[GO:0006275],P


In [8]:
df_complete = df_complete.merge(df_terms, on="id", how="left")
df_complete.head()

,db,id,name,full_name,organism,taxon,gen,PE,SV,sequence,length,go_terms,aspect
0,sp,A0A0C5B5G6,MOTSC_HUMAN,Mitochondrial-derived peptide MOTS-c,Homo sapiens,9606,MT-RNR1,1,1,MRWQEMGYIFYPRKLR,16,[GO:0001649],P
1,sp,A0A0C5B5G6,MOTSC_HUMAN,Mitochondrial-derived peptide MOTS-c,Homo sapiens,9606,MT-RNR1,1,1,MRWQEMGYIFYPRKLR,16,[GO:0033687],P
2,sp,A0A0C5B5G6,MOTSC_HUMAN,Mitochondrial-derived peptide MOTS-c,Homo sapiens,9606,MT-RNR1,1,1,MRWQEMGYIFYPRKLR,16,[GO:0005615],C
3,sp,A0A0C5B5G6,MOTSC_HUMAN,Mitochondrial-derived peptide MOTS-c,Homo sapiens,9606,MT-RNR1,1,1,MRWQEMGYIFYPRKLR,16,[GO:0005634],C
4,sp,A0A0C5B5G6,MOTSC_HUMAN,Mitochondrial-derived peptide MOTS-c,Homo sapiens,9606,MT-RNR1,1,1,MRWQEMGYIFYPRKLR,16,[GO:0005739],C


In [9]:
df_complete.columns

Index(['db', 'id', 'name', 'full_name', 'organism', 'taxon', 'gen', 'PE', 'SV',
       'sequence', 'length', 'go_terms', 'aspect'],
      dtype='object')

In [10]:
# Columns to aggregate
cols_a_listar = ['go_terms', 'aspect']

# Metadata
fix_cols = [c for c in df_complete.columns if c not in cols_a_listar and c != 'id']

# Dict with aggregation rules
rules = {col: list for col in cols_a_listar}
rules.update({col: 'first' for col in fix_cols})

df_final = df_complete.groupby('id').agg(rules).reset_index()

df_final = df_final[df_complete.columns]

df_final

,db,id,name,full_name,organism,taxon,gen,PE,SV,sequence,length,go_terms,aspect
0,sp,A0A023FBW4,E1142_AMBCJ,Evasin P1142,Amblyomma cajennense,34607,None,3,1,MTSHGAVKIAIFAVIALHSIFECLSKPQILQRTDHSTDSDWDPQMC...,97,[[GO:0019958]],[F]
1,sp,A0A023FBW7,EV546_AMBCJ,Evasin P546,Amblyomma cajennense,34607,None,3,1,MKVLLYIAASCLMLLALNVSAENTQQEEEDYDYGTDTCPFPVLANK...,118,[[GO:0019957]],[F]
2,sp,A0A023FDY8,EV974_AMBCJ,Evasin P974,Amblyomma cajennense,34607,None,1,1,MKVLLCIAASCLMLLALNVSAENTQQEEQDYDYGTDTCPFPVLANK...,118,[[GO:0019957]],[F]
3,sp,A0A023FF81,E1126_AMBCJ,Evasin P1126,Amblyomma cajennense,34607,None,3,1,MTSHSAVRIAIFAVIALHSIFECLSKPQILQRTDKSTDSEWDPQTC...,90,[[GO:0019958]],[F]
4,sp,A0A023FFB5,EV983_AMBCJ,Evasin P983,Amblyomma cajennense,34607,None,3,2,MKASFCVIASCLVVFALKGTAEDTGTEDDFDYGNTGCPFPVLGNYK...,119,[[GO:0019957]],[F]
...,...,...,...,...,...,...,...,...,...,...,...,...,...
82399,sp,X2JI34,GAO_TANPA,Germacrene A hydroxylase,Tanacetum parthenium,127999,None,1,1,MALSLTTSIALATILFFVYKFATRSKSTKNSLPEPWRLPIIGHMHH...,488,"[[GO:0106223], [GO:0051762]]","[F, P]"
82400,sp,X4Y2L4,LHYAL_HIRNI,Hyaluronoglucuronidase,Hirudo nipponia,42736,None,1,1,MKEIAVTIDDKNVIASVSESFHGVAFDASLFSPKGLWSFVDITSPK...,489,"[[GO:0033906], [GO:0030214]]","[F, P]"
82401,sp,X5JA13,SECAA_ARATH,Exocyst complex component SEC10a,Arabidopsis thaliana,3702,SEC10a,1,1,MTEGIRARGPRSSSVNSVPLILDIEDFKGDFSFDALFGNLVNDLLP...,825,"[[GO:0070062], [GO:0005829], [GO:0005515], [GO...","[C, C, F, C, C, C, C, P]"
82402,sp,X5JB51,SECAB_ARATH,Exocyst complex component SEC10b,Arabidopsis thaliana,3702,SEC10b,1,1,MTERIRARGPRSSSVNSVPLILDIEDFKGDFSFDALFGNLVNDLLP...,829,"[[GO:0070062], [GO:0005829], [GO:0005886], [GO...","[C, C, C, C, C, C, P, F]"


In [11]:
# Check if the length of sequences matches the total rows of the dataframe
all(df_final['length'] == df_final['sequence'].str.len())

True

In [15]:
# Save the final dataframe to a CSV file
df_final.to_csv(train_path + "train_data_prepared.csv", index=False)